# 02 — Identifier structure

**Do the 29-bit identifiers encode node addresses?**

The hypothesis (`bikecan/ids.py`): every identifier reads as four bytes
`PP TT AA BB` — priority, message type, and two node fields. If `AA` and `BB`
really are node addresses, every frame can be attributed to two physical
components, and the search space for everything else collapses.

This notebook tests it. Nothing here is settled yet.

In [ ]:
import sys
from pathlib import Path

# The project is not installed as a package, so put the repo root on the path.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

from bikecan import dbc, discover, experiment, ids, session

SESSIONS = REPO / "data" / "sessions"

In [ ]:
s = session.load(SESSIONS / "legacy")
can = s.can
print(f"{len(can):,} frames, {can['id_hex'].nunique()} identifiers")

## The field values actually observed

A field that takes only a handful of values across a whole corpus is being used
as an enumeration, which is what a priority, a message type and a node address
all look like. A field spread across many values would argue against the reading.

In [ ]:
for column in ["priority", "msg_type", "node_a", "node_b"]:
    values = sorted(can[column].unique())
    print(f"{column:9} {len(values):2} values: " + " ".join(f"{v:02X}" for v in values))

## Traffic by node pair

If the reading is right, each row here is two physical components talking to
each other.

In [ ]:
discover.node_summary(can)

## The strongest evidence: request and response

A zero-length frame carries no data, so it is asking rather than telling. If the
answer comes back on the same node pair with a different message type, the node
fields are real.

In [ ]:
pairs = discover.request_response(can, window_ms=10)
pairs

In [ ]:
if len(pairs):
    print(f"{len(pairs)} request/response pairs")
    print(f"share the node pair: {pairs['same_node_pair'].all()}")
    print(f"gaps (ms): {sorted(pairs['gap_ms'])}")
else:
    print("no zero-length frames in this session -- nothing to pair")

## The competing reading

`AA` might itself be two nibbles: note how `16`/`06`, `36`/`03` and `46`/`09`
pair up suspiciously. Both readings survive until a session disproves one.

In [ ]:
rows = []
for id_hex in sorted(can["id_hex"].unique()):
    decoded = ids.decode(int(id_hex, 16))
    high, low = decoded.nibbles
    rows.append({
        "id_hex": id_hex,
        "priority": f"{decoded.priority:02X}",
        "msg_type": f"{decoded.msg_type:02X}",
        "node_a": f"{decoded.node_a:02X}",
        "node_b": f"{decoded.node_b:02X}",
        "a_nibbles": f"{high:X}|{low:X}",
        "low_nibble_matches_b": low == decoded.node_b,
    })
pd.DataFrame(rows)

## What would settle it

Neither reading can be confirmed from traffic patterns alone. It needs a
stimulus experiment: record with `bikelog`, mark the instant a component is
operated, and see which node pair reacts. That is notebook 03.

Record what you conclude — and the evidence for it — in `dbc/nodes.md`.